In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime
from pathlib import Path

import folium
import geopandas as gpd
import rasterio
import ulmo

from rasterio.features import shapes
from shapely.geometry import shape, Polygon, Point
from shapely import unary_union

from mintpy.utils import utils as ut, readfile
# from mintpy.objects import timeseries

# Input Data

## HyP3 Interferograms

In [ ]:
intf_dir = Path('~/hyp3_data/clipped_data/')
intf_list = list(intf_dir.glob('*unw_phase_clipped.tif'))

### Get Sentinel-1 Acquisition Dates

In [ ]:
def get_unique_dates(file_paths):
    """Extracts unique observation dates from a list of Sentinel-1 file paths.
    
    Args:
        file_paths (list): List of PosixPath or string file paths.
    
    Returns:
        list: Sorted list of unique observation dates (YYYY-MM-DD).
    """
    date_pattern = re.compile(r'_(\d{8})T')  # Matches YYYYMMDD before 'T'
    unique_dates = set()

    for file_path in file_paths:
        file_name = Path(file_path).name  # Get only the filename
        dates = date_pattern.findall(file_name)  # Extract dates
        dates = pd.to_datetime(dates, format='%Y%m%d').date.tolist()  # Convert to list of date objects
        unique_dates.update(dates)  # Add to set

    return sorted(unique_dates)  # Return sorted list of unique dates


In [ ]:
s1_dates = get_unique_dates(intf_list)
print(s1_dates)

### Get Valid Data Polygon

In [ ]:
def get_valid_data_polygon(intf_list):
    list_parts = []
    for _, intf in enumerate(intf_list):
        with rasterio.open(intf) as src:
            # Get the valid data mask directly
            mask = src.read_masks(1)
            
            # Convert mask to polygons
            shapes_generator = shapes(mask.astype('uint8'), mask=mask, transform=src.transform)
            
            # Convert to Shapely geometries
            polygons = [shape(geom) for geom, value in shapes_generator]
            
            for polygon in polygons:
                list_interiors = []
                
                for interior in polygon.interiors:
                    p = interior
    
                    if p.area > 0.0001:
                        list_interiors.append(interior)
                temp_pol = Polygon(polygon.exterior.coords, holes=list_interiors)
                list_parts.append(temp_pol)
    
    # Merge all extracted polygons into a single geometry
    valid_area = unary_union(list_parts)
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(geometry=[valid_area], crs=src.crs)
    
    # Convert to Lat/Lon (EPSG:4326)
    gdf = gdf.to_crs(epsg=4326)
        
    return gdf

In [ ]:
gdf = get_valid_data_polygon(intf_list)

In [ ]:
# Visualize gdf polygon

# Get centroid
centroid = gdf.geometry.centroid.iloc[0]
    
# Create folium map centered at the centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=8)

# Add the GeoDataFrame geometry as a GeoJSON layer
folium.GeoJson(gdf.to_json(), name="Valid Data Area").add_to(m)

m

## MintPy Timeseries

### Get Dates

In [ ]:
geo_dir = Path('~/TEST/snowsar/MintPy/geo')

In [ ]:
ts_file = geo_dir.joinpath('geo_timeseries_ERA5.h5')

In [ ]:
date_list = readfile.get_slice_list(ts_file)

In [ ]:
s1_dates = [datetime.strptime(d.split("-")[1], "%Y%m%d").date() for d in dates]

In [ ]:
date_list

### Get Polygon

In [ ]:
def get_valid_data_polygon_from_array(array, north, south, east, west, X_STEP, Y_STEP):
    """
    Extracts the largest valid area polygon from an array with NaNs, using known geographic bounds and step sizes.

    Args:
        array (2D np.array): Input array with NaNs where data is missing.
        north (float): Maximum Y coordinate (top boundary).
        south (float): Minimum Y coordinate (bottom boundary).
        east (float): Maximum X coordinate (right boundary).
        west (float): Minimum X coordinate (left boundary).
        X_STEP (float): Pixel width (resolution in X direction).
        Y_STEP (float): Pixel height (resolution in Y direction, negative for north-down orientation).

    Returns:
        gpd.GeoDataFrame: GeoDataFrame containing the largest valid region as a polygon.
    """

    # Get array shape
    rows, cols = array.shape

    # Define affine transformation (maps pixel coordinates to real-world coordinates)
    transform = rasterio.transform.from_origin(west, north, X_STEP, -Y_STEP)

    # Convert array to binary mask (1 = valid data, 0 = NaN)
    mask = np.isfinite(array).astype(np.uint8)

    # Extract polygons from the binary mask
    shapes_generator = shapes(mask, mask=mask, transform=transform)
    polygons = [shape(geom) for geom, value in shapes_generator if value == 1]
    list_parts = []
    for polygon in polygons:
        list_interiors = []
        
        for interior in polygon.interiors:
            p = interior
    
            if p.area > 0.0001:
                list_interiors.append(interior)
        temp_pol = Polygon(polygon.exterior.coords, holes=list_interiors)
        list_parts.append(temp_pol)
    
    # Merge all extracted polygons into a single geometry
    valid_area = unary_union(list_parts)
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(geometry=[valid_area], crs="EPSG:4326")

    return gdf

In [ ]:
ts_arr, attr = readfile.read(ts_file, datasetName=f'{date_list[0]}')
s, n, w, e = ut.four_corners(attr)
x_step = float(attr['X_STEP'])
y_step = float(attr['Y_STEP'])

In [ ]:
gdf = get_valid_data_polygon_from_array(ts_arr, n, s, e, w, x_step, y_step)

In [ ]:
# Visualize gdf polygon

# Get centroid
centroid = gdf.geometry.centroid.iloc[0]
    
# Create folium map centered at the centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=8)

# Add the GeoDataFrame geometry as a GeoJSON layer
folium.GeoJson(gdf.to_json(), name="Valid Data Area").add_to(m)

m

# Get SNOTEL Sites

In [ ]:
#This is the latest CUAHSI API endpoint
wsdlurl = 'https://hydroportal.cuahsi.org/Snotel/cuahsi_1_1.asmx?WSDL'

In [ ]:
sites = ulmo.cuahsi.wof.get_sites(wsdlurl)

In [ ]:
sites_df = pd.DataFrame.from_dict(sites, orient='index').dropna()
sites_df['geometry'] = [Point(float(loc['longitude']), float(loc['latitude'])) for loc in sites_df['location']]
sites_df = sites_df.drop(columns='location')
sites_df = sites_df.astype({"elevation_m":float})
sites_gdf = gpd.GeoDataFrame(sites_df, crs='EPSG:4326')

In [ ]:
idx = sites_gdf.intersects(gdf.iloc[0].geometry)
snotel_sites = sites_gdf.loc[idx]

In [ ]:
# Get the centroid of the geometry to center the map
centroid = gdf.geometry.centroid.iloc[0]

# Create folium map centered at the centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=8)

# Add the GeoDataFrame geometry as a GeoJSON layer
folium.GeoJson(gdf.to_json(), name="Valid Data Area").add_to(m)

# Add points to the map
for _, row in snotel_sites.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],  # Folium uses (lat, lon)
        popup='Name: '+row['name']+' - Code: '+row['code'],  # Display name and code when clicked
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(m)

# Display map
m

# Get SNOTEL data and make plots

In [ ]:
start_date = datetime(2020,12,1)
end_date = datetime(2021,4,1)
# end_date = datetime.today()

In [ ]:
def fetch_snotel_data(snotel_sites, wsdlurl, start_date, end_date, reference_date="12-01"):
    """Fetches SNOTEL data for multiple sites and calculates cumulative daily change.
    
    Args:
        snotel_sites (pd.DataFrame): DataFrame containing site codes and names.
        wsdlurl (str): The WSDL URL for data retrieval.
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        reference_date (str): Reference date for computing 'days since' (default: "12-01").
    
    Returns:
        dict: Dictionary with site names as keys and processed DataFrames as values.
    """
    results = {}

    for site_code, site_name, site_loc in zip(snotel_sites["code"], snotel_sites["name"], snotel_sites["geometry"]):
        try:
            # Fetch site data
            site_values = ulmo.cuahsi.wof.get_values(wsdlurl, site_code, 'WTEQ_H', start=start_date, end=end_date)
            
            # Convert to DataFrame
            values_df = pd.DataFrame.from_dict(site_values['values'])

            # Remove lower-quality records
            values_df = values_df[values_df['quality_control_level_code'] == '1']

            # Drop unnecessary columns
            values_df = values_df.drop(columns=['qualifiers', 'censor_code', 'method_id', 'method_code', 
                                                'source_code', 'quality_control_level_code', 'datetime'])

            # Convert to datetime and numeric
            values_df["date_time_utc"] = pd.to_datetime(values_df["date_time_utc"])
            values_df["value"] = pd.to_numeric(values_df["value"], errors='coerce').astype('float32')

            # Convert inches to cm
            values_df["value_cm"] = values_df["value"] * 2.54

            # Filter values at 6 AM
            value_at_6am = values_df[values_df["date_time_utc"].dt.hour == 6].copy()

            # Compute days since reference date
            reference_year = pd.to_datetime(start_date).year
            reference_datetime = pd.to_datetime(f"{reference_year}-{reference_date}")
            value_at_6am["days_since_reference"] = (value_at_6am["date_time_utc"] - reference_datetime).dt.days
            value_at_6am["site_loc"] = site_loc

            # Store results
            results[site_name] = value_at_6am[["date_time_utc", "days_since_reference", "value_cm", "site_loc"]]

        except Exception as e:
            print(f"Skipping site {site_code} ({site_name}) due to error: {e}")

    return results


In [ ]:
def plot_snotel_data(results, s1_dates):
    """Plots in situ SWE (cm) and the mean/std deviation of delta SWE on s1_dates.
    
    Args:
        results (dict): Dictionary with site names as keys and processed DataFrames as values.
        s1_dates (list): List of Sentinel-1 observation dates as datetime.date objects.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # Two side-by-side plots

    # Convert Sentinel-1 dates to "days since December 1st"
    reference_date = min(s1_dates)
    s1_days_since_reference = [(date - reference_date).days for date in s1_dates]

    # --- Plot 1: SWE time series with Sentinel-1 observation markers ---
    for site_name, data in results.items():
        axes[0].plot(data["days_since_reference"], data["value_cm"], linestyle='-', label=site_name)

    # Add vertical dashed lines for Sentinel-1 observation dates
    for day in s1_days_since_reference:
        axes[0].axvline(day, color='k', linestyle='--', alpha=0.7, linewidth=1)

    # axes[0].axhline(0, color='k', linestyle='--', linewidth=2)
    axes[0].set_xlabel("Days Since December 1st")
    axes[0].set_ylabel("In Situ SWE (cm)")
    axes[0].set_ylim(0, 100)
    axes[0].set_title("Daily SWE at 6 AM (cm)")
    axes[0].grid(True, alpha=0.25)

    # --- Compute delta SWE (daily change) only on s1_dates ---
    all_deltas, all_dates = [], []
    
    for data in results.values():
        # Filter data to only include rows corresponding to s1_dates
        filtered_data = data[data["date_time_utc"].dt.date.isin(s1_dates)].copy()
    
        # Sort by date (to ensure chronological order)
        filtered_data = filtered_data.sort_values(by="date_time_utc")
    
        # Compute delta SWE between consecutive s1_dates
        filtered_data["delta_swe_cm"] = filtered_data["value_cm"].diff()
    
        # Store results
        all_dates.extend(filtered_data["date_time_utc"][1:])  # Skip first date (since diff() produces NaN for first row)
        all_deltas.extend(filtered_data["delta_swe_cm"][1:])
    
    # Convert to NumPy arrays
    all_dates = np.array(all_dates, dtype="datetime64[D]")  # Convert to NumPy datetime array
    all_deltas = np.array(all_deltas)
    
    # Compute mean and standard deviation of delta SWE **between** s1_dates
    unique_s1_dates = np.unique(all_dates)
    mean_deltas = np.array([np.mean(all_deltas[all_dates == d]) for d in unique_s1_dates])
    std_deltas = np.array([np.std(all_deltas[all_dates == d]) for d in unique_s1_dates])

    # --- Plot 2: Mean and Std Dev of delta SWE on Sentinel-1 dates with error bars ---
    axes[1].errorbar(unique_s1_dates, mean_deltas, yerr=std_deltas, fmt='o-', color='b', ecolor='b', capsize=3)

    # Label each data point with its corresponding date
    for i, date in enumerate(unique_s1_dates):
        date_str = pd.to_datetime(str(date)).strftime('%Y-%m-%d')

    # axes[1].axhline(0, color='k', linestyle='--', linewidth=2)
    axes[1].set_xlabel("Date")
    axes[1].set_ylabel("Δ SWE (cm)")
    axes[1].set_ylim(-4, 8)
    axes[1].set_title("Mean and Std Dev of Δ SWE on Sentinel-1 Dates")
    axes[1].grid(True)
    axes[1].set_xticks(s1_dates)
    axes[1].tick_params(axis='x', rotation=45)

    # Adjust layout and show the plot
    plt.tight_layout()
    plt.show()

In [ ]:
swe_data = fetch_snotel_data(snotel_sites, wsdlurl, start_date, end_date)

In [ ]:
plot_snotel_data(swe_data, s1_dates)

In [ ]:
import pickle

def save_snotel_data_pickle(results, filename="snotel_data.pkl"):
    """Saves SNOTEL data dictionary as a pickle file."""
    with open(filename, "wb") as f:
        pickle.dump(results, f)

In [ ]:
save_snotel_data_pickle(swe_data)